In [1]:
# Install first if needed:
# pip install paddleocr paddlepaddle opencv-python

from paddleocr import PaddleOCR
import cv2
import re
import json

# 1. Load OCR model
ocr = PaddleOCR(use_angle_cls=True, lang="en")

# 2. Image path
image_path = "../Data/receipt.png"

# 3. Image preprocessing
image = cv2.imread(image_path)

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, None, fx=1.5, fy=1.5)
gray = cv2.GaussianBlur(gray, (3, 3), 0)

processed_path = "processed_receipt.jpg"
cv2.imwrite(processed_path, gray)

# 4. OCR Detection + Recognition
result = ocr.ocr(processed_path, cls=True)

# 5. Extract text
lines = []

for page in result:
    for box, text_info in page:
        text = text_info[0]
        confidence = text_info[1]
        lines.append(text)

full_text = "\n".join(lines)

print("===== OCR TEXT =====")
print(full_text)

# 6. Post-processing / field extraction
def extract_receipt_data(text):
    data = {
        "merchant": None,
        "date": None,
        "subtotal": None,
        "vat": None,
        "total": None,
        "items": []
    }

    lines = text.split("\n")

    # merchant usually first line
    if lines:
        data["merchant"] = lines[0]

    # date
    date_pattern = r"\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2}"
    date_match = re.search(date_pattern, text)
    if date_match:
        data["date"] = date_match.group()

    # subtotal
    subtotal_match = re.search(r"subtotal\s*[:\-]?\s*(\d+\.\d{2})", text, re.IGNORECASE)
    if subtotal_match:
        data["subtotal"] = float(subtotal_match.group(1))

    # VAT / tax
    vat_match = re.search(r"(vat|tax)\s*@?\d*%?\s*[:\-]?\s*(\d+\.\d{2})", text, re.IGNORECASE)
    if vat_match:
        data["vat"] = float(vat_match.group(2))

    # total
    total_match = re.search(r"total\s*[:\-]?\s*(\d+\.\d{2})", text, re.IGNORECASE)
    if total_match:
        data["total"] = float(total_match.group(1))

    # items: simple pattern → Item name followed by price
    for i in range(len(lines) - 1):
        name = lines[i]
        price = lines[i + 1]

        if re.match(r"^[A-Za-z ]+$", name) and re.match(r"^\d+\.\d{2}$", price):
            data["items"].append({
                "name": name,
                "price": float(price)
            })

    return data

receipt_data = extract_receipt_data(full_text)

print("\n===== JSON OUTPUT =====")
print(json.dumps(receipt_data, indent=4))

[2026/05/12 23:19:30] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\Yonas/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\Yonas/.paddleocr/whl\\rec\\en\\en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_len